First Few Steps are Run on Colab as the protein - ligand decoupling code uses Colab libraries

### *BOLTZ 2 Google Colab Interface*

2025-06-13 by Aarshit Mittal

This notebook provides a simple way to harness the power of [BOLTZ 2](https://github.com/jwohlwend/boltz) (MIT license) for:
- High-accuracy protein structure prediction
- Protein-ligand binding affinity calculation  
- Protein-protein complex modeling
- Mutation impact analysis
- Multi-variant comparison
- Structure quality assessment

---

In [1]:
#@title Install Dependencies
import os
import sys

# --- 1. Define required versions ---
# This makes it easy to update versions in the future
REQUIRED_TORCH_VERSION = "2.4.0"
REQUIRED_PL_VERSION = "2.5.0"

# A flag to determine if we need to run the installation
needs_install = False

# --- 2. Check if the correct versions are already installed ---
try:
    import torch
    import pytorch_lightning as pl

    # Check if the installed versions match our requirements.
    # We use .startswith() to ignore suffixes like "+cu118"
    torch_ok = torch.__version__.startswith(REQUIRED_TORCH_VERSION)
    pl_ok = pl.__version__.startswith(REQUIRED_PL_VERSION)

    if torch_ok and pl_ok:
        print(f"✓ Correct versions already installed.")
        print(f"  - PyTorch: {torch.__version__}")
        print(f"  - PyTorch-Lightning: {pl.__version__}")
        print("Skipping installation.")
    else:
        print("Mismatched versions detected. Reinstallation is required.")
        if not torch_ok:
            print(f"  - Found PyTorch {torch.__version__}, but require {REQUIRED_TORCH_VERSION}")
        if not pl_ok:
            print(f"  - Found PyTorch-Lightning {pl.__version__}, but require {REQUIRED_PL_VERSION}")
        needs_install = True

except ImportError:
    # This block runs if one of the libraries isn't installed at all
    print("Required libraries not found. Installation is required.")
    needs_install = True

# --- 3. Run installation and restart ONLY if needed ---
if needs_install:
    print("\nUninstalling existing PyTorch to avoid conflicts...")
    # Using sys.executable ensures we use the correct pip
    !{sys.executable} -m pip uninstall torch torchvision torchaudio -y -q

    print("Installing specified library versions...")
    !{sys.executable} -m pip install torch=={REQUIRED_TORCH_VERSION} torchvision==0.19.0 --index-url https://download.pytorch.org/whl/cu118 -q
    !{sys.executable} -m pip install pytorch-lightning=={REQUIRED_PL_VERSION} boltz py3Dmol -q
    print("Installation complete.")

    # Crucial step: Restart the runtime to load the new libraries
    print("\nIMPORTANT: Runtime is restarting to load new versions. Please wait...")
    os.kill(os.getpid(), 9)

✓ Correct versions already installed.
  - PyTorch: 2.4.0+cu118
  - PyTorch-Lightning: 2.5.0
Skipping installation.


In [2]:
#@title Verify Installation
import torch
import pytorch_lightning as pl
import boltz
import py3Dmol

print("✅ Dependencies loaded successfully!")
print(f"✓ PyTorch version: {torch.__version__}")
print(f"✓ PyTorch-Lightning version: {pl.__version__}")
print(f"✓ BOLTZ installed")
print(f"✓ py3Dmol installed")
print("-" * 30)
print(f"✓ GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Not available'}")

✅ Dependencies loaded successfully!
✓ PyTorch version: 2.4.0+cu118
✓ PyTorch-Lightning version: 2.5.0
✓ BOLTZ installed
✓ py3Dmol installed
------------------------------
✓ GPU: Tesla P100-PCIE-16GB


In [3]:
#@title Import Libraries
import os
import sys
import subprocess
import json
import tempfile
import yaml
import shutil
from pathlib import Path
import time
from datetime import datetime
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import py3Dmol
from google.colab import files
from IPython.display import display, HTML
import torch

# Configure environment
os.makedirs('output', exist_ok=True)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
gpu_info = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'
print(f"Device: {device} ({gpu_info})")

Device: cuda (Tesla P100-PCIE-16GB)


In [6]:
#@title Core BOLTZ Interface

class BOLTZPredictor:
    """Main interface for BOLTZ 2 predictions"""

    def __init__(self, output_dir: str = "output"):
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(exist_ok=True)

    def run_prediction(self, input_data: dict, job_name: str,
                      use_msa: bool = True, verbose: bool = False) -> dict:
        """
        Execute BOLTZ prediction

        Args:
            input_data: YAML-compatible input dictionary
            job_name: Unique identifier for this job
            use_msa: Whether to use MSA server
            verbose: Print detailed output

        Returns:
            Dictionary containing results and file paths
        """
        # Create temporary YAML file
        with tempfile.NamedTemporaryFile(mode='w', suffix='.yaml', delete=False) as f:
            yaml.dump(input_data, f)
            yaml_path = f.name

        # Build command
        output_path = self.output_dir / job_name
        cmd = ["boltz", "predict", yaml_path, "--out_dir", str(output_path)]

        if use_msa:
            cmd.append("--use_msa_server")

        try:
            # Execute prediction
            process = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                                     stderr=subprocess.PIPE, text=True)
            stdout, stderr = process.communicate()

            if process.returncode != 0:
                return {"success": False, "error": stderr or stdout}

            # Parse results
            result_dirs = list(output_path.glob("boltz_results_*"))

            files = {"pdb": [], "cif": [], "json": []}
            for result_dir in result_dirs:
                files["pdb"].extend(list(result_dir.rglob("*.pdb")))
                files["cif"].extend(list(result_dir.rglob("*.cif")))
                files["json"].extend(list(result_dir.rglob("*.json")))

            # Load JSON results
            results = {}
            for json_file in files["json"]:
                try:
                    with open(json_file) as f:
                        results[json_file.name] = json.load(f)
                except Exception as e:
                    if verbose:
                        print(f"Warning: Could not read {json_file.name}: {e}")

            return {
                "success": True,
                "output_dir": str(output_path),
                "files": files,
                "results": results,
                "stdout": stdout if verbose else None
            }

        except Exception as e:
            return {"success": False, "error": str(e)}
        finally:
            Path(yaml_path).unlink(missing_ok=True)

    def extract_metrics(self, results: dict) -> dict:
        """Extract key metrics from BOLTZ results"""
        metrics = {}

        for filename, data in results.items():
            if "confidence" in filename and isinstance(data, dict):
                metrics.update({
                    "confidence_score": data.get("confidence_score", 0),
                    "ptm": data.get("ptm", 0),
                    "iptm": data.get("iptm", 0),
                    "plddt": data.get("complex_plddt", 0)
                })
            elif "affinity" in filename and isinstance(data, dict):
                # FIXED: Properly interpret affinity results
                # affinity_pred_value is log10(IC50 in μM) - this is what BOLTZ outputs
                log_ic50_uM = data.get("affinity_pred_value", 0)

                # Convert log10(IC50 in μM) to IC50 in μM
                ic50_uM = 10 ** log_ic50_uM

                # Convert to nM (1 μM = 1000 nM)
                ic50_nM = ic50_uM * 1000

                # Convert to M for proper pIC50 calculation
                ic50_M = ic50_uM * 1e-6

                # Calculate standard pIC50 (negative log of IC50 in molar units)
                # pIC50 = -log10(IC50 in M)
                pic50 = -np.log10(ic50_M) if ic50_M > 0 else 0

                # Calculate binding free energy in kcal/mol
                # Using the formula from BOLTZ documentation: (6 - affinity) * 1.364
                # Note: this assumes affinity_pred_value is in the range where 6 corresponds to 1 μM
                delta_g_kcal = (6 - log_ic50_uM) * 1.364

                # Approximate Kd from IC50 (Kd ≈ IC50/2 for competitive inhibitors)
                # This is a rough approximation!
                kd_uM = ic50_uM / 2
                kd_nM = kd_uM * 1000
                kd_M = kd_uM * 1e-6

                # Calculate pKd from Kd in molar units
                pkd = -np.log10(kd_M) if kd_M > 0 else 0

                metrics.update({
                    # Raw BOLTZ output
                    "boltz_affinity_value": log_ic50_uM,  # This is log10(IC50 in μM)

                    # IC50 values
                    "log_ic50_uM": log_ic50_uM,
                    "ic50_uM": ic50_uM,
                    "ic50_nM": ic50_nM,
                    "pic50": pic50,  # Standard pIC50

                    # Kd approximations
                    "kd_uM": kd_uM,
                    "kd_nM": kd_nM,
                    "pkd": pkd,

                    # Energy
                    "delta_g_kcal": delta_g_kcal,

                    # Binding probability
                    "affinity_prob": data.get("affinity_probability_binary", 0)
                })

        return metrics

# Initialize predictor
predictor = BOLTZPredictor()

In [7]:
#@title Visualization Functions

def visualize_structure(structure_path: Path, width: int = 800, height: int = 600):
    """Display 3D molecular structure"""

    with open(structure_path, 'r') as f:
        content = f.read()

    file_format = 'pdb' if str(structure_path).endswith('.pdb') else 'cif'

    view = py3Dmol.view(width=width, height=height)
    view.addModel(content, file_format)
    view.setStyle({'cartoon': {'color': 'spectrum'}})
    view.setBackgroundColor('white')
    view.zoomTo()

    return view

def plot_confidence_distribution(plddt_file: Path) -> plt.Figure:
    """Plot per-residue confidence distribution"""

    data = np.load(plddt_file)
    plddt_values = data['plddt'] * 100  # Convert to percentage

    fig, ax = plt.subplots(figsize=(10, 6))

    # Histogram
    n, bins, patches = ax.hist(plddt_values, bins=50, alpha=0.7,
                               color='#2E86AB', edgecolor='black')

    # Color code by confidence level
    for i, patch in enumerate(patches):
        if bins[i] < 50:
            patch.set_facecolor('#D32F2F')  # Red - Very low
        elif bins[i] < 70:
            patch.set_facecolor('#F57C00')  # Orange - Low
        elif bins[i] < 90:
            patch.set_facecolor('#FBC02D')  # Yellow - Confident
        else:
            patch.set_facecolor('#388E3C')  # Green - High

    ax.axvline(plddt_values.mean(), color='red', linestyle='--', linewidth=2,
               label=f'Mean: {plddt_values.mean():.1f}%')

    ax.set_xlabel('pLDDT Score (%)', fontsize=12)
    ax.set_ylabel('Number of Residues', fontsize=12)
    ax.set_title('Per-Residue Confidence Distribution', fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    return fig

In [13]:
# Predict Protein-Ligand Binding function

def protein_ligand_binding(protein_sequence, ligand_smiles, job_name, binding_pocket,template_path):
    # Prepare input
    contacts = [["A", i] for i in binding_pocket]
    input_data = {
        "version": 1,
        "sequences": [
            {
                "protein": {
                    "id": "A",
                    "sequence": protein_sequence.upper().strip()
                }
            },
            {
                "ligand": {
                    "id": "C",
                    "smiles": ligand_smiles.strip()
                }
            }
        ],
        "constraints": [
            {
                "pocket": {
                    "binder": "C",
                    "contacts": contacts
                }
            }
        ],
        "properties": [
            {
                "affinity": {
                    "binder": "C",
                    "target": "A"
                }
            }
        ],
        "template": {
            "complex_cif": template_path
        }
    }

    print(f"Protein: {len(protein_sequence)} residues")
    print(f"Ligand: {ligand_smiles}")

    # Run prediction
    result = predictor.run_prediction(input_data, job_name)

    if result["success"]:
        metrics = predictor.extract_metrics(result["results"])

    # Check if affinity was calculated
        if "boltz_affinity_value" in metrics:
            print("\nBinding Affinity Results:")
            print("=" * 50)

        # Show raw model output
            print(f"\nRaw BOLTZ Output:")
            print(f"  affinity_pred_value: {metrics['boltz_affinity_value']:.3f}")
            print(f"  (This is log10(IC50 in μM) according to BOLTZ docs)")
            print(f"  Binding Probability: {metrics['affinity_prob']:.3f} ({metrics['affinity_prob']*100:.1f}%)")

        # Show converted values
            print(f"\nConverted Values:")
            print(f"  IC50: {metrics['ic50_nM']:.1f} nM ({metrics['ic50_uM']:.3f} μM)")
            print(f"  pIC50 (standard): {metrics['pic50']:.2f}")
            print(f"  ΔG (binding): {metrics['delta_g_kcal']:.2f} kcal/mol")

        # Approximate Kd values
            print(f"\nApproximate Kd (assuming competitive inhibition):")
            print(f"  Kd ≈ {metrics['kd_nM']:.1f} nM ({metrics['kd_uM']:.3f} μM)")
            print(f"  pKd ≈ {metrics['pkd']:.2f}")

        # Affinity classification based on IC50
            ic50_nm = metrics['ic50_nM']
            if ic50_nm < 10:
                affinity_class = "Very Strong (< 10 nM)"
                emoji = "🟢"
            elif ic50_nm < 100:
                affinity_class = "Strong (10-100 nM)"
                emoji = "🟢"
            elif ic50_nm < 1000:
                affinity_class = "Moderate (0.1-1 μM)"
                emoji = "🟡"
            elif ic50_nm < 10000:
                affinity_class = "Weak (1-10 μM)"
                emoji = "🟠"
            else:
                affinity_class = "Very Weak (> 10 μM)"
                emoji = "🔴"

            print(f"\nAffinity Classification (based on IC50): {emoji} {affinity_class}")

        # Important notes
            print("\nNotes:")
            print("- BOLTZ outputs log10(IC50 in μM), not pKd")
            print("- ΔG calculated using BOLTZ formula: (6 - log_ic50) * 1.364 kcal/mol")
            print("- Kd values are approximated as IC50/2 (valid for competitive inhibitors)")
            print("- For accurate Kd determination, experimental Ki measurements are needed")
        else:
            print("\nNo affinity results found.")
            print("Check that BOLTZ was run with affinity prediction enabled.")

    # Show structure quality metrics
        print("\nStructure Quality Metrics:")
        print(f"  Confidence: {metrics.get('confidence_score', 0):.3f}")
        print(f"  pTM: {metrics.get('ptm', 0):.3f}")
        print(f"  pLDDT: {metrics.get('plddt', 0):.3f}")

    # Visualize complex
        if result["files"]["cif"]:
            view = visualize_structure(result["files"]["cif"][0])
        # Enhanced visualization for protein-ligand complex
            view.setStyle({'cartoon': {'color': 'spectrum'}})
            view.addStyle({'hetflag': True}, {'stick': {'radius': 0.3, 'colorscheme': 'greenCarbon'}})
            view.zoomTo()
            view.show()

            print(f"\nComplex structure saved: {result['files']['cif'][0]}")
        elif result["files"]["pdb"]:
            view = visualize_structure(result["files"]["pdb"][0])
            view.setStyle({'cartoon': {'color': 'spectrum'}})
            view.addStyle({'hetflag': True}, {'stick': {'radius': 0.3, 'colorscheme': 'greenCarbon'}})
            view.zoomTo()
            view.show()

            print(f"\nComplex structure saved: {result['files']['pdb'][0]}")
    else:
        print(f"Prediction failed: {result['error']}")

# Ligand SMILE strucutres

Ligand_83="NS(=O)(=O)c1ccc(Cc2c(-c3ccc(F)c(F)c3)nn(-c3nc(C(=O)O)cs3)c2-c2ccc(-c3cccc(F)c3)c(F)c2)cc1F"
Ligand_94="NS(=O)(=O)c1ccc(Cc2c(-c3cccc(-c4ccc(C#CC5(O)CCC5)cc4)c3)nn(-c3nc(C(=O)O)cs3)c2-c2cccc(F)c2)cc"
Ligand_172="Cc1n[nH]c2c(F)cc(C(=O)Nc3ccc4c(c3)nc(CN3CCC(C[C@@H]5CCO5)CC3)n4C[C@@H]3CCO3)cc12"

# GLP-1R Protein descriptors

GLP1R_ecd_ranges = list(range(24,140)) + list(range(202,228)) + list(range(291,306)) + list(range(371,384)) # ECD
GLP1R_tm_ranges =  list (range(140,161)) + list (range(176,202)) + list (range(228,252)) + list (range(266,291)) + list(range(306,329)) + list(range(349,371)) + list(range(384,405)) #TM
GLP1R_binding_pocket=GLP1R_ecd_ranges+GLP1R_tm_ranges
GLP1R_sequence="MAGAPGLLRLALLLLGMVGRAGPRPQGATVSLWETVQKWREYRRQCQRSLTEDPPPATDLFCNRTFDEYACWPDGEPGSFVNVSCPWYLPWASSVPQGHVYRFCTAEGLWLQKDNSSLPWRDLSECEESKRGERSSPEEQLLFLYIIYTVGYALSFSALVIASAILLGFRHLHCTRNYIHLNLFASFILRALSVFIKDAALKWMYSTAAQQHQWDGLLSYQDSLSCRLVFLLMQYCVAANYYWLLVEGVYLYTLLAFSVLSEQWIFRLYVSIGWGVPLLFVVPWGIVKYLYEDEGCWTRNSNMNYWLIIRLPILFAIGVNFLIFVRVICIVVSKLKANLMCKTDIKCRLAKSTLTLIPLLGTHEVIFAFVMDEHARGTLRFIKLFTELSFTSFQGLMVAILYCFVNNEVQLEFRKSWERWRLEHLHIQRDSSMKPLKCPTSSLSSGATAGSSMYTATCQASCS"
GLP1R_template_path="/kaggle/input/glp1r-ligand-free/ligand_free_processed.cif"


protein_ligand_binding(GLP1R_sequence,Ligand_83, "GLP1R-LIGAND83",GLP1R_binding_pocket,GLP1R_template_path)
protein_ligand_binding(GLP1R_sequence,Ligand_94, "GLP1R-LIGAND94",GLP1R_binding_pocket,GLP1R_template_path)
protein_ligand_binding(GLP1R_sequence,Ligand_172, "GLP1R-LIGAND172",GLP1R_binding_pocket,GLP1R_template_path)

# GIPR Protein descriptors

GIPR_ecd_ranges = list(range(22,139)) + list(range(190,218)) + list(range(279,294)) + list(range(363,378)) # ECD
GIPR_tm_ranges =  list (range(139,162)) + list (range(170,190)) + list (range(218,243)) + list (range(255,279)) + list(range(294,320)) + list(range(342,363)) + list(range(379,399)) #TM
GIPR_binding_pocket=GIPR_ecd_ranges+GIPR_tm_ranges
GIPR_sequence="MTTSPILQLLLRLSLCGLLLQRAETGSKGQTAGELYQRWERYRRECQETLAAAEPPSGLACNGSFDMYVCWDYAAPNATARASCPWYLPWHHHVAAGFVLRQCGSDGQWGLWRDHTQCENPEKNEAFLDQRLILERLQVMYTVGYSLSLATLLLALLILSLFRRLHCTRNYIHINLFTSFMLRAAAILSRDRLLPRPGPYLGDQALALWNQALAACRTAQIVTQYCVGANYTWLLVEGVYLHSLLVLVGGSEEGHFRYYLLLGWGAPALFVIPWVIVRYLYENTQCWERNEVKAIWWIIRTPILMTILINFLIFIRILGILLSKLRTRQMRCRDYRLRLARSTLTLVPLLGVHEVVFAPVTEEQARGALRFAKLGFEIFLSSFQGFLVSVLYCFINKEVQSEIRRGWHHCRLRRSLGEEQRQLPERAFRALPSGSGPGEVPTSRGLSSGTLPGPGNEASRELESYC" #466 sequence length
GIPR_template_path="/kaggle/input/glp1r-ligand-free/ligand_free_fully_processed__7FIN GIP.cif"


protein_ligand_binding(GIPR_sequence,Ligand_83, "GIP-LIGAND83",GIPR_binding_pocket,GIPR_template_path)
protein_ligand_binding(GIPR_sequence,Ligand_94, "GIP-LIGAND94",GIPR_binding_pocket,GIPR_template_path)
protein_ligand_binding(GIPR_sequence,Ligand_172, "GIP-LIGAND172",GIPR_binding_pocket,GIPR_template_path)

# GCGRR Protein descriptors


GCGR_ecd_ranges = list(range(26,137)) + list(range(199,226)) + list(range(286,305)) + list(range(370,382)) # ECD
GCGR_tm_ranges =  list (range(137,162)) + list (range(174,199)) + list (range(226,250)) + list (range(264,286)) + list(range(304,327)) + list(range(351,370)) + list(range(382,403)) #TM
GCGR_binding_pocket=GCGR_ecd_ranges+GCGR_tm_ranges
GCGR_sequence="MPPCQPQRPLLLLLLLLACQPQVPSAQVMDFLFEKWKLYGDQCHHNLSLLPPPTELVCNRTFDKYSCWPDTPANTTANISCPWYLPWHHKVQHRFVFKRCGPDGQWVRGPRGQPWRDASQCQMDGEEIEVQKEVAKMYSSFQVMYTVGYSLSLGALLLALAILGGLSKLHCTRNAIHANLFASFVLKASSVLVIDGLLRTRYSQKIGDDLSVSTWLSDGAVAGCRVAAVFMQYGIVANYCWLLVEGLYLHNLLGLATLPERSFFSLYLGIGWGAPMLFVVPWAVVKCLFENVQCWTSNDNMGFWWILRFPVFLAILINFFIFVRIVQLLVAKLRARQMHHTDYKFRLAKSTLTLIPLLGVHEVVFAFVTDEHAQGTLRSAKLFFDLFLSSFQGLLVAVLYCFLNKEVQSELRRRWHRWRLGKVLWEERNTSNHRASSSPGHGPPSKELQFGRGGGSQDSSAETPLAGGLPRLAESPF" #477 sequence length - https://www.uniprot.org/uniprotkb/P47871/entry
GCGR_template_path="/kaggle/input/glp1r-ligand-free/ligand_free_fully_processed__8YW5 GCGR.cif"

protein_ligand_binding(GCGR_sequence,Ligand_83, "GCGR-LIGAND83",GCGR_binding_pocket,GCGR_template_path)
protein_ligand_binding(GCGR_sequence,Ligand_94, "GCGR-LIGAND94",GCGR_binding_pocket,GCGR_template_path)
protein_ligand_binding(GCGR_sequence,Ligand_172, "GCGR-LIGAND172",GCGR_binding_pocket,GCGR_template_path)








Protein: 463 residues
Ligand: NS(=O)(=O)c1ccc(Cc2c(-c3ccc(F)c(F)c3)nn(-c3nc(C(=O)O)cs3)c2-c2ccc(-c3cccc(F)c3)c(F)c2)cc1F

Binding Affinity Results:

Raw BOLTZ Output:
  affinity_pred_value: -0.697
  (This is log10(IC50 in μM) according to BOLTZ docs)
  Binding Probability: 0.193 (19.3%)

Converted Values:
  IC50: 200.8 nM (0.201 μM)
  pIC50 (standard): 6.70
  ΔG (binding): 9.14 kcal/mol

Approximate Kd (assuming competitive inhibition):
  Kd ≈ 100.4 nM (0.100 μM)
  pKd ≈ 7.00

Affinity Classification (based on IC50): 🟡 Moderate (0.1-1 μM)

Notes:
- BOLTZ outputs log10(IC50 in μM), not pKd
- ΔG calculated using BOLTZ formula: (6 - log_ic50) * 1.364 kcal/mol
- Kd values are approximated as IC50/2 (valid for competitive inhibitors)
- For accurate Kd determination, experimental Ki measurements are needed

Structure Quality Metrics:
  Confidence: 0.794
  pTM: 0.756
  pLDDT: 0.758


3Dmol.js failed to load for some reason. Please check your browser console for error messages.


Complex structure saved: output/GLP1R-LIGAND83/boltz_results_tmp_u51nc92/predictions/tmp_u51nc92/tmp_u51nc92_model_0.cif
